# Website Silver Layer: Fact Tables & dbt Model Generation

Transformed bronze website data into silver-layer fact tables, validated data quality and referential integrity, and generated dbt staging and marts model files for downstream analytics.

## 1. Building Silver Fact Tables from Bronze Website Data

Transformed three bronze tables into deduplicated silver fact tables:

* `fact_web_session` — sessions with source, medium, device, and campaign attribution
* `fact_web_page_view` — page views linked to sessions via `session_id`
* `fact_web_event` — interaction events (search, property view, booking, purchase) linked to sessions

Used a `latest_record()` helper to deduplicate by business key, keeping the most recent record per `updated_at` and `_airbyte_extracted_at`. Cast timestamps to proper types before writing to the `voyage.silver` schema.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ---------------------------------------------------------
# Helper for keeping the latest version of each record
# ---------------------------------------------------------

def latest_record(df, business_key):
    window = (
        Window
        .partitionBy(business_key)
        .orderBy(
            F.col("updated_at").desc(),
            F.col("_airbyte_extracted_at").desc(),
        )
    )

    return (
        df
        .withColumn("_row_number", F.row_number().over(window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )


# ---------------------------------------------------------
# 1. Web Sessions  (bronze --> silver fact table)
# ---------------------------------------------------------

bronze_web_sessions = spark.table("voyage.bronze.website_sessions")

web_sessions = (
    bronze_web_sessions
    .withColumn("session_start", F.to_timestamp("session_start"))
    .withColumn("session_end",   F.to_timestamp("session_end"))
    .withColumn("updated_at",     F.to_timestamp("updated_at"))
)

silver_web_sessions = latest_record(web_sessions, "session_id").select(
    "campaign_id",
    "session_id",
    "tenant_id",
    "user_id",
    "device",
    "country",
    "landing_page",
    "medium",
    "source",
    "session_start",
    "session_end",
    "updated_at",
)

(
    silver_web_sessions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.fact_web_session")
)


# ---------------------------------------------------------
# 2. Web Page Views  (bronze → silver fact table)
# ---------------------------------------------------------

bronze_web_page_views = spark.table("voyage.bronze.website_page_views")

web_page_views = (
    bronze_web_page_views
    .withColumn("page_viewed_at", F.to_timestamp("page_viewed_at"))
    .withColumn("updated_at",      F.to_timestamp("updated_at"))
)

silver_web_page_views = latest_record(web_page_views, "page_view_id").select(
    "page_view_id",
    "session_id",
    "tenant_id",
    "page_path",
    "page_title",
    "referrer",
    "duration_seconds",
    "page_viewed_at",
    "updated_at",
)

(
    silver_web_page_views.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.fact_web_page_view")
)


# ---------------------------------------------------------
# 3. Web Events  (bronze → silver fact table)
# ---------------------------------------------------------

bronze_web_events = spark.table("voyage.bronze.website_events")

web_events = (
    bronze_web_events
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
    .withColumn("event_value",      F.col("event_value").cast("decimal(14,2)"))
    .withColumn("updated_at",        F.to_timestamp("updated_at"))
)

silver_web_events = latest_record(web_events, "event_id").select(
    "event_id",
    "session_id",
    "tenant_id",
    "event_name",
    "event_value",
    "page_path",
    "event_timestamp",
    "updated_at",
)

(
    silver_web_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.fact_web_event")
)


print("Website silver fact tables created successfully.")

## 2. Silver Layer Data Quality Verification

Ran four quality checks across all three silver fact tables:

* Row counts — confirmed each table loaded with expected volume
* Schema inspection — displayed column names, types, and nullability
* Null checks — validated that key columns (IDs, timestamps, tenant) contained no nulls
* Duplicate key checks — ensured business keys (`session_id`, `page_view_id`, `event_id`) were unique

In [0]:
# ---------------------------------------------------------
# Silver website verification — basic quality checks
#   1. Row counts
#   2. Schemas
#   3. Null checks on key columns
#   4. Duplicate key checks
#   (Referential integrity checks are in the next cell)
# ---------------------------------------------------------

print("=" * 70)
print("1. ROW COUNTS")
print("=" * 70)

for tbl in [
    "voyage.silver.fact_web_session",
    "voyage.silver.fact_web_page_view",
    "voyage.silver.fact_web_event",
]:
    cnt = spark.table(tbl).count()
    print(f"  {tbl:50s}  {cnt:>10,} rows")

print()
print("=" * 70)
print("2. SCHEMAS")
print("=" * 70)

for tbl in [
    "voyage.silver.fact_web_session",
    "voyage.silver.fact_web_page_view",
    "voyage.silver.fact_web_event",
]:
    print(f"\n  --- {tbl} ---")
    for f in spark.table(tbl).schema.fields:
        print(f"    {f.name:25s}  {str(f.dataType):20s}  nullable={f.nullable}")

print()
print("=" * 70)
print("3. NULL CHECKS ON KEY COLUMNS")
print("=" * 70)

null_checks = [
    ("voyage.silver.fact_web_session",     "session_id"),
    ("voyage.silver.fact_web_session",     "tenant_id"),
    ("voyage.silver.fact_web_session",     "session_start"),
    ("voyage.silver.fact_web_page_view",   "page_view_id"),
    ("voyage.silver.fact_web_page_view",   "session_id"),
    ("voyage.silver.fact_web_page_view",   "tenant_id"),
    ("voyage.silver.fact_web_page_view",   "page_viewed_at"),
    ("voyage.silver.fact_web_event",      "event_id"),
    ("voyage.silver.fact_web_event",      "session_id"),
    ("voyage.silver.fact_web_event",      "tenant_id"),
    ("voyage.silver.fact_web_event",      "event_timestamp"),
]

for tbl, col in null_checks:
    null_cnt = spark.table(tbl).filter(F.col(col).isNull()).count()
    status = "PASS" if null_cnt == 0 else f"FAIL ({null_cnt:,} nulls)"
    print(f"  {tbl:50s}  {col:20s}  {status}")

print()
print("=" * 70)
print("4. DUPLICATE KEY CHECKS")
print("=" * 70)

dup_checks = [
    ("voyage.silver.fact_web_session",   "session_id"),
    ("voyage.silver.fact_web_page_view", "page_view_id"),
    ("voyage.silver.fact_web_event",    "event_id"),
]

for tbl, key in dup_checks:
    dup_cnt = (
        spark.table(tbl)
        .groupBy(key)
        .agg(F.count("*").alias("cnt"))
        .filter(F.col("cnt") > 1)
        .count()
    )
    status = "PASS" if dup_cnt == 0 else f"FAIL ({dup_cnt:,} dup keys)"
    print(f"  {tbl:50s}  {key:20s}  {status}")

print()
print("Verification complete.")

## 3. Referential Integrity Validation

Checked that foreign-key relationships held across the silver layer with zero expected orphans:

1. `fact_web_session` → `dim_tenant` (on `tenant_id`)
2. `fact_web_page_view` → `fact_web_session` (on `session_id`)
3. `fact_web_event` → `fact_web_session` (on `session_id`)
4. `fact_web_session` → `dim_campaign` (on `campaign_id`, non-null only — organic/direct traffic is excluded)

In [0]:
# ---------------------------------------------------------
# Silver website validation -- referential integrity only
#   Silver = clean, deduplicated analytical source (no KPIs)
#
# Checks (expected orphan count = 0 for each):
#   1. fact_web_session   -> dim_tenant        (tenant_id)
#   2. fact_web_page_view -> fact_web_session  (session_id)
#   3. fact_web_event     -> fact_web_session  (session_id)
#   4. fact_web_session  -> dim_campaign      (campaign_id, non-null only)
# ---------------------------------------------------------

print("=" * 70)
print("SILVER WEBSITE -- REFERENTIAL INTEGRITY VALIDATION")
print("=" * 70)

all_pass = True

# --- 1. session -> tenant ------------------------------------------
session_tenant_orphans = (
    spark.table("voyage.silver.fact_web_session")
    .select("tenant_id")
    .distinct()
    .join(
        spark.table("voyage.silver.dim_tenant").select("tenant_id").distinct(),
        "tenant_id",
        "left_anti",
    )
    .count()
)
status_1 = "PASS" if session_tenant_orphans == 0 else f"FAIL ({session_tenant_orphans:,} orphans)"
if session_tenant_orphans != 0:
    all_pass = False
print(f"  1. fact_web_session   -> dim_tenant         {status_1}")

# --- 2. page_view -> session ---------------------------------------
page_view_session_orphans = (
    spark.table("voyage.silver.fact_web_page_view")
    .select("session_id")
    .distinct()
    .join(
        spark.table("voyage.silver.fact_web_session").select("session_id").distinct(),
        "session_id",
        "left_anti",
    )
    .count()
)
status_2 = "PASS" if page_view_session_orphans == 0 else f"FAIL ({page_view_session_orphans:,} orphans)"
if page_view_session_orphans != 0:
    all_pass = False
print(f"  2. fact_web_page_view -> fact_web_session    {status_2}")

# --- 3. event -> session -------------------------------------------
event_session_orphans = (
    spark.table("voyage.silver.fact_web_event")
    .select("session_id")
    .distinct()
    .join(
        spark.table("voyage.silver.fact_web_session").select("session_id").distinct(),
        "session_id",
        "left_anti",
    )
    .count()
)
status_3 = "PASS" if event_session_orphans == 0 else f"FAIL ({event_session_orphans:,} orphans)"
if event_session_orphans != 0:
    all_pass = False
print(f"  3. fact_web_event     -> fact_web_session    {status_3}")

# --- 4. session campaign_id -> dim_campaign (non-null only) --------
sessions_with_campaign = (
    spark.table("voyage.silver.fact_web_session")
    .filter(F.col("campaign_id").isNotNull())
)

campaign_orphans = (
    sessions_with_campaign
    .select("campaign_id")
    .distinct()
    .join(
        spark.table("voyage.silver.dim_campaign").select("campaign_id").distinct(),
        "campaign_id",
        "left_anti",
    )
    .count()
)
status_4 = "PASS" if campaign_orphans == 0 else f"FAIL ({campaign_orphans:,} orphans)"
if campaign_orphans != 0:
    all_pass = False
print(f"  4. fact_web_session   -> dim_campaign       {status_4}  (non-null campaign_ids only)")

print("=" * 70)
if all_pass:
    print("RESULT: ALL CHECKS PASSED -- 0 orphans across all 4 relationships")
else:
    print("RESULT: SOME CHECKS FAILED -- review orphan counts above")
print("=" * 70)

## 4. Silver Table Column Listing for dbt Staging

Printed the column names of each silver fact table to help write dbt staging model `SELECT` statements. This kept the staging models in sync with the actual silver schema.

In [0]:
# ---------------------------------------------------------
# Column names for dbt staging references
# ---------------------------------------------------------

silver_web_tables = [
    "voyage.silver.fact_web_session",
    "voyage.silver.fact_web_page_view",
    "voyage.silver.fact_web_event",
]

for tbl in silver_web_tables:
    cols = spark.table(tbl).columns
    print(f"-- {tbl} ({len(cols)} columns)")
    print(",\n".join(f"    {c}" for c in cols))
    print()

## 5. dbt Staging Model Generation

Generated dbt staging layer files for all three silver fact tables:

* `sources.yml` — source definitions with column tests (not_null, unique, relationships)
* `stg_web_sessions.sql` / `.yml` — staged sessions with campaign FK relationship
* `stg_web_page_views.sql` / `.yml` — staged page views with session FK relationship
* `stg_web_events.sql` / `.yml` — staged events with session FK and event value casting

Each SQL model selected directly from the silver source and was materialized as a view.

In [0]:
# ---------------------------------------------------------
# dbt staging model files for website silver tables
# Copy each block below into the corresponding file.
# ---------------------------------------------------------

print("=" * 70)
print("1. dbt/models/staging/sources.yml  (add to existing Silver source)")
print("=" * 70)
print("""
  - name: fact_web_session
    description: "Deduplicated website sessions from bronze"
    columns:
      - name: session_id
        description: "Unique session identifier"
        tests: [not_null, unique]
      - name: tenant_id
        tests: [not_null]
      - name: session_start
        tests: [not_null]
      - name: session_end
        tests: [not_null]
      - name: source
        tests: [not_null]
      - name: medium
        tests: [not_null]
      - name: campaign_id
        tests:
          - relationships:
              to: ref('stg_campaigns')
              field: campaign_id
              # null values are allowed (organic/direct traffic)

  - name: fact_web_page_view
    description: "Deduplicated website page views from bronze"
    columns:
      - name: page_view_id
        tests: [not_null, unique]
      - name: tenant_id
        tests: [not_null]
      - name: session_id
        tests:
          - not_null
          - relationships:
              to: ref('stg_web_sessions')
              field: session_id
      - name: page_viewed_at
        tests: [not_null]

  - name: fact_web_event
    description: "Deduplicated website events from bronze"
    columns:
      - name: event_id
        tests: [not_null, unique]
      - name: tenant_id
        tests: [not_null]
      - name: session_id
        tests:
          - not_null
          - relationships:
              to: ref('stg_web_sessions')
              field: session_id
      - name: event_timestamp
        tests: [not_null]
      - name: event_name
        tests: [not_null]
""")

print()
print("=" * 70)
print("2. dbt/models/staging/stg_web_sessions.sql")
print("=" * 70)
print("""
{{ config(materialized='view') }}

select
    campaign_id,
    session_id,
    tenant_id,
    user_id,
    device,
    country,
    landing_page,
    medium,
    source,
    session_start,
    session_end,
    updated_at
from {{ source('silver', 'fact_web_session') }}
""")

print()
print("=" * 70)
print("3. dbt/models/staging/stg_web_sessions.yml")
print("=" * 70)
print("""
version: 2

models:
  - name: stg_web_sessions
    description: "Cleaned and renamed website sessions from silver"
    columns:
      - name: session_id
        description: "Unique session identifier"
        tests: [not_null, unique]
      - name: tenant_id
        tests: [not_null]
      - name: user_id
        description: "User identifier"
      - name: device
        description: "Device type (mobile, desktop, tablet)"
      - name: country
        description: "Country code"
      - name: landing_page
        description: "First page viewed in the session"
      - name: medium
        tests: [not_null]
      - name: source
        tests: [not_null]
      - name: campaign_id
        description: "Campaign identifier, nullable for organic/direct traffic"
        tests:
          - relationships:
              to: ref('stg_campaigns')
              field: campaign_id
      - name: session_start
        tests: [not_null]
      - name: session_end
        tests: [not_null]
      - name: updated_at
""")

print()
print("=" * 70)
print("4. dbt/models/staging/stg_web_page_views.sql")
print("=" * 70)
print("""
{{ config(materialized='view') }}

select
    page_view_id,
    session_id,
    tenant_id,
    page_path,
    page_title,
    referrer,
    duration_seconds,
    page_viewed_at,
    updated_at
from {{ source('silver', 'fact_web_page_view') }}
""")

print()
print("=" * 70)
print("5. dbt/models/staging/stg_web_page_views.yml")
print("=" * 70)
print("""
version: 2

models:
  - name: stg_web_page_views
    description: "Cleaned and renamed website page views from silver"
    columns:
      - name: page_view_id
        description: "Unique page view identifier"
        tests: [not_null, unique]
      - name: session_id
        tests:
          - not_null
          - relationships:
              to: ref('stg_web_sessions')
              field: session_id
      - name: tenant_id
        tests: [not_null]
      - name: page_path
        description: "URL path of the viewed page"
      - name: page_title
        description: "Title of the viewed page"
      - name: referrer
        description: "Referrer URL"
      - name: duration_seconds
        description: "Time spent on page in seconds"
      - name: page_viewed_at
        tests: [not_null]
      - name: updated_at
""")

print()
print("=" * 70)
print("6. dbt/models/staging/stg_web_events.sql")
print("=" * 70)
print("""
{{ config(materialized='view') }}

select
    event_id,
    session_id,
    tenant_id,
    event_name,
    cast(event_value as decimal(14,2)) as event_value,
    page_path,
    event_timestamp,
    updated_at
from {{ source('silver', 'fact_web_event') }}
""")

print()
print("=" * 70)
print("7. dbt/models/staging/stg_web_events.yml")
print("=" * 70)
print("""
version: 2

models:
  - name: stg_web_events
    description: "Cleaned and renamed website events from silver"
    columns:
      - name: event_id
        description: "Unique event identifier"
        tests: [not_null, unique]
      - name: session_id
        tests:
          - not_null
          - relationships:
              to: ref('stg_web_sessions')
              field: session_id
      - name: tenant_id
        tests: [not_null]
      - name: event_name
        description: "Event type (search_availability, view_property, begin_booking, purchase)"
        tests: [not_null]
      - name: event_value
        description: "Event payload value, nullable"
      - name: page_path
        description: "URL path where the event was triggered"
      - name: event_timestamp
        tests: [not_null]
      - name: updated_at
""")

print()
print("All dbt staging files generated.")

## 6. dbt Marts Model Generation

Generated dbt marts layer files for website analytics:

* `fct_web_session.sql` / `.yml` — session-level fact table with engagement and conversion metrics (page views, events, searches, property views, booking starts, conversions) sourced from `int_web_session_metrics`
* `fct_website_daily.sql` / `.yml` — daily aggregation by tenant, source, medium, and campaign with computed conversion rate, enriched with campaign name/channel/type from `stg_campaigns`

Both marts models were materialized as tables for query performance.

In [0]:
# ---------------------------------------------------------
# dbt marts model files for website analytics
# Copy each block below into the corresponding file.
# ---------------------------------------------------------

print("=" * 70)
print("1. dbt/models/marts/fct_web_session.sql")
print("=" * 70)
print("""
{{ config(materialized='table') }}

select
    session_id,
    tenant_id,
    user_id,
    session_start,
    session_end,
    source,
    medium,
    campaign_id,
    device,
    country,
    landing_page,
    page_view_count,
    event_count,
    search_count,
    property_view_count,
    booking_start_count,
    conversion_count,
    conversion_value,
    engaged_session,
    session_duration_seconds,
    updated_at
from {{ ref('int_web_session_metrics') }}
""")

print()
print("=" * 70)
print("2. dbt/models/marts/fct_web_session.yml")
print("=" * 70)
print("""
version: 2

models:
  - name: fct_web_session
    description: "Session-level fact table with engagement and conversion metrics"
    columns:
      - name: session_id
        description: "Unique session identifier"
        tests: [not_null, unique]
      - name: tenant_id
        tests: [not_null]
      - name: user_id
        description: "User identifier"
      - name: session_start
        tests: [not_null]
      - name: session_end
        tests: [not_null]
      - name: source
        tests: [not_null]
      - name: medium
        tests: [not_null]
      - name: campaign_id
        description: "Campaign identifier, nullable for organic/direct traffic"
      - name: device
        description: "Device type (mobile, desktop, tablet)"
      - name: country
        description: "Country code"
      - name: landing_page
        description: "First page viewed in the session"
      - name: page_view_count
        description: "Number of page views in the session"
      - name: event_count
        description: "Total events fired in the session"
      - name: search_count
        description: "Number of search_availability events"
      - name: property_view_count
        description: "Number of view_property events"
      - name: booking_start_count
        description: "Number of begin_booking events"
      - name: conversion_count
        description: "Number of purchase events"
      - name: conversion_value
        description: "Sum of purchase event values"
      - name: engaged_session
        description: "1 if session had 2+ page views or any events, else 0"
      - name: session_duration_seconds
        description: "Duration from session_start to session_end in seconds"
      - name: updated_at
""")

print()
print("=" * 70)
print("3. dbt/models/marts/fct_website_daily.sql")
print("=" * 70)
print("""
{{ config(materialized='table') }}

with session_metrics as (

    select
        date(session_start) as date,
        tenant_id,
        source,
        medium,
        campaign_id,

        count(*)                             as sessions,
        sum(engaged_session)                 as engaged_sessions,
        sum(page_view_count)                 as page_views,
        sum(event_count)                     as events,
        sum(search_count)                    as searches,
        sum(property_view_count)             as property_views,
        sum(booking_start_count)            as booking_starts,
        sum(conversion_count)                as conversions,
        round(sum(conversion_value), 2)      as conversion_value,

        try_divide(
            sum(conversion_count),
            count(*)
        ) as conversion_rate

    from {{ ref('fct_web_session') }}
    group by
        date(session_start),
        tenant_id,
        source,
        medium,
        campaign_id

),

campaign_lookup as (

    select
        campaign_id,
        campaign_name,
        channel,
        campaign_type
    from {{ ref('stg_campaigns') }}

)

select
    sm.date,
    sm.tenant_id,
    sm.source,
    sm.medium,
    sm.campaign_id,

    sm.sessions,
    sm.engaged_sessions,
    sm.page_views,
    sm.events,
    sm.searches,
    sm.property_views,
    sm.booking_starts,
    sm.conversions,
    sm.conversion_value,
    sm.conversion_rate,

    cl.campaign_name,
    cl.channel,
    cl.campaign_type

from session_metrics sm
left join campaign_lookup cl
    on sm.campaign_id = cl.campaign_id
""")

print()
print("=" * 70)
print("4. dbt/models/marts/fct_website_daily.yml")
print("=" * 70)
print("""
version: 2

models:
  - name: fct_website_daily
    description: "Daily website metrics aggregated by tenant, source, medium, and campaign"
    columns:
      - name: date
        description: "Calendar date derived from session_start"
        tests: [not_null]
      - name: tenant_id
        tests: [not_null]
      - name: source
        tests: [not_null]
      - name: medium
        tests: [not_null]
      - name: campaign_id
        description: "Campaign identifier, nullable for organic/direct traffic"
      - name: sessions
        description: "Total number of sessions"
      - name: engaged_sessions
        description: "Sessions with 2+ page views or at least one event"
      - name: page_views
        description: "Total page views across all sessions"
      - name: events
        description: "Total events fired across all sessions"
      - name: searches
        description: "Total search_availability events"
      - name: property_views
        description: "Total view_property events"
      - name: booking_starts
        description: "Total begin_booking events"
      - name: conversions
        description: "Total purchase events"
      - name: conversion_value
        description: "Sum of purchase event values"
      - name: conversion_rate
        description: "Conversions divided by sessions"
      - name: campaign_name
        description: "Campaign name from stg_campaigns"
      - name: channel
        description: "Campaign channel from stg_campaigns"
      - name: campaign_type
        description: "Campaign type from stg_campaigns"
""")

print()
print("All dbt marts files generated.")

## 7. Final reconciliation: silver vs. dbt mart

Built the silver fact tables and generated the dbt marts models in the previous cells. Before calling this pipeline done, ran a reconciliation query to make sure nothing drifted between layers. The query compared five metrics — sessions, page views, events, conversions, and conversion value — between the silver source tables and `voyage.dbt_dev.fct_web_session`. If all five matched, the dbt transformations preserved the data correctly. If any metric was off, something in the staging or marts logic dropped or duplicated rows and needed fixing before this notebook was complete.

In [0]:
%sql
with silver_sessions as (
    select count(*) as sessions
    from voyage.silver.fact_web_session
),

silver_page_views as (
    select count(*) as page_views
    from voyage.silver.fact_web_page_view
),

silver_events as (
    select count(*) as events
    from voyage.silver.fact_web_event
),

silver_conversions as (
    select
        count(*) as conversions,
        round(sum(coalesce(event_value, 0)), 2) as conversion_value
    from voyage.silver.fact_web_event
    where event_name = 'purchase'
),

session_mart as (
    select
        count(*) as sessions,
        sum(page_view_count) as page_views,
        sum(event_count) as events,
        sum(conversion_count) as conversions,
        round(sum(conversion_value), 2) as conversion_value
    from voyage.dbt_dev.fct_web_session
)

select
    case
        when ss.sessions = sm.sessions
         and sp.page_views = sm.page_views
         and se.events = sm.events
         and sc.conversions = sm.conversions
         and sc.conversion_value = sm.conversion_value
        then 'PASS'
        else 'FAIL'
    end as reconciliation_status,

    ss.sessions as silver_sessions,
    sm.sessions as mart_sessions,

    sp.page_views as silver_page_views,
    sm.page_views as mart_page_views,

    se.events as silver_events,
    sm.events as mart_events,

    sc.conversions as silver_conversions,
    sm.conversions as mart_conversions,

    sc.conversion_value as silver_conversion_value,
    sm.conversion_value as mart_conversion_value

from silver_sessions ss
cross join silver_page_views sp
cross join silver_events se
cross join silver_conversions sc
cross join session_mart sm

## 8. Marketing consistency: orphan campaign check

The reconciliation test in the previous cell confirmed row counts matched between silver and the mart. Then checked the campaign dimension side — sessions in `fct_web_session` that carried a `campaign_id` but had no matching record in `stg_campaigns`. These orphan campaigns would break any join to campaign metadata downstream and cause missing campaign names in the daily report. Ran a left join between the mart and `stg_campaigns` on `campaign_id`, filtering for non-null campaign IDs where the campaign table came back null. Expected zero orphans — anything above zero meant a campaign was tracked in the session data but never made it into the campaigns dimension.

In [0]:
%sql
select count(*) as orphan_campaign_sessions
from voyage.dbt_dev.fct_web_session s
left join voyage.dbt_dev.stg_campaigns c
    on s.campaign_id = c.campaign_id
where s.campaign_id is not null
  and c.campaign_id is null